In [ ]:
# --- Setup ---
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from country_sweep import COUNTRY_SITES

In [ ]:
# --- Load prices for all countries in one year ---
csv_path = 'ember_prices.csv'
year = 2023
countries = sorted(COUNTRY_SITES.keys())

raw = pd.read_csv(csv_path, parse_dates=['Datetime (UTC)'])
raw = raw[raw['Datetime (UTC)'].dt.year == year]
raw = raw[['ISO3 Code', 'Datetime (UTC)', 'Price (EUR/MWhe)']]
raw = raw.rename(columns={'ISO3 Code': 'country',
                          'Datetime (UTC)': 't',
                          'Price (EUR/MWhe)': 'price'})
raw['price'] = raw['price'].clip(lower=0.0)

wide = raw.pivot_table(index='t', columns='country', values='price')
print(wide.shape)
wide.describe().T[['mean', 'std', 'min', '50%', 'max']]

(8760, 29)


,mean,std,min,50%,max
country,,,,,
AUT,263.276342,135.197139,0.00,225.580,838.73
BEL,244.875987,133.962905,0.00,219.810,871.00
BGR,252.825339,131.561643,0.17,224.870,936.33
CHE,281.639848,128.218647,0.00,251.300,871.61
CZE,247.438196,137.510026,0.00,215.730,871.00
DEU,236.522158,141.106649,0.00,208.190,867.85
DNK,215.695731,146.059636,0.00,189.870,871.00
ESP,167.505266,69.464028,0.00,163.545,700.00
EST,192.812728,135.663071,0.00,160.735,4000.00


In [42]:
# --- Distribution of prices per country ---
fig = go.Figure()
for c in countries:
    if c in wide.columns:
        fig.add_trace(go.Box(y=wide[c], name=c))
fig.update_layout(
    title=f'Price distribution by country, {year}',
    yaxis_title='Price [EUR/MWh]',
    height=500, showlegend=False,
)
fig.show()

In [43]:
# --- Monthly average prices per country ---
monthly = wide.resample('ME').mean()

fig = go.Figure()
for c in countries:
    if c in monthly.columns:
        fig.add_trace(go.Scatter(x=monthly.index, y=monthly[c],
                                 mode='lines', name=c))
fig.update_layout(
    title=f'Monthly average price by country, {year}',
    yaxis_title='Price [EUR/MWh]',
    height=500, hovermode='x unified',
)
fig.show()

In [44]:
# --- Hourly shape: mean price by hour of day, per country ---
by_hour = wide.groupby(wide.index.hour).mean()

fig = go.Figure()
for c in countries:
    if c in by_hour.columns:
        fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour[c],
                                 mode='lines+markers', name=c))
fig.update_layout(
    title=f'Mean price by hour of day, {year}',
    xaxis_title='Hour (UTC)',
    yaxis_title='Price [EUR/MWh]',
    height=500, hovermode='x unified',
)
fig.show()

In [45]:
# --- Mean price by local hour of day, per country ---
# Each country has its own timezone; the CSV gives UTC. Convert before
# grouping so the daily shape reflects the local demand cycle.

COUNTRY_TZ = {
    'AUT': 'Europe/Vienna',
    'BEL': 'Europe/Brussels',
    'CHE': 'Europe/Zurich',
    'CZE': 'Europe/Prague',
    'DEU': 'Europe/Berlin',
    'DNK': 'Europe/Copenhagen',
    'ESP': 'Europe/Madrid',
    'EST': 'Europe/Tallinn',
    'FIN': 'Europe/Helsinki',
    'FRA': 'Europe/Paris',
    'GRC': 'Europe/Athens',
    'HUN': 'Europe/Budapest',
    'ITA': 'Europe/Rome',
    'LTU': 'Europe/Vilnius',
    'LUX': 'Europe/Luxembourg',
    'LVA': 'Europe/Riga',
    'NLD': 'Europe/Amsterdam',
    'NOR': 'Europe/Oslo',
    'POL': 'Europe/Warsaw',
    'PRT': 'Europe/Lisbon',
    'ROU': 'Europe/Bucharest',
    'SVK': 'Europe/Bratislava',
    'SVN': 'Europe/Ljubljana',
    'SWE': 'Europe/Stockholm',
}


def local_hourly(series, tz):
    if series.index.tz is None:
        idx = series.index.tz_localize('UTC').tz_convert(tz)
    else:
        idx = series.index.tz_convert(tz)
    return pd.Series(series.values, index=idx).groupby(idx.hour).mean()


fig = go.Figure()
for c in countries:
    if c in wide.columns and c in COUNTRY_TZ:
        by_local = local_hourly(wide[c], COUNTRY_TZ[c])
        fig.add_trace(go.Scatter(x=by_local.index, y=by_local.values,
                                 mode='lines+markers', name=c))
fig.update_layout(
    title=f'Mean price by local hour of day, {year}',
    xaxis_title='Hour (local)',
    yaxis_title='Price [EUR/MWh]',
    xaxis=dict(dtick=2),
    height=500, hovermode='x unified',
)
fig.show()

In [37]:
# --- Cross-country correlation ---
corr = wide.corr()

fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.update_layout(title=f'Price correlation across countries, {year}',
                  height=700)
fig.show()

In [38]:
# --- Summary table ---
summary = pd.DataFrame({
    'mean': wide.mean(),
    'median': wide.median(),
    'std': wide.std(),
    'p10': wide.quantile(0.10),
    'p90': wide.quantile(0.90),
    'p99': wide.quantile(0.99),
    'hours_above_200': (wide > 200).sum(),
    'hours_below_20': (wide < 20).sum(),
})
summary['peak_to_median'] = summary['p99'] / summary['median']
summary = summary.sort_values('mean', ascending=False)
summary

,mean,median,std,p10,p90,p99,hours_above_200,hours_below_20,peak_to_median
country,,,,,,,,,
ITA,122.732384,123.700,18.582439,100.190,143.750,165.3003,6,4,1.336300
SRB,117.998384,116.755,36.163364,77.069,153.881,243.6137,210,64,2.086538
ALB,117.127879,111.070,53.837503,66.000,180.000,307.5700,208,76,2.769155
IRL,114.428209,105.720,54.236904,66.399,171.954,312.9461,522,319,2.960141
MNE,113.243040,112.000,42.706154,70.010,160.000,232.1191,221,316,2.072492
MKD,111.635050,108.100,50.159784,61.090,165.880,277.0800,400,379,2.563182
HUN,109.025154,104.710,57.798992,34.387,172.331,300.2232,462,743,2.867187
ROU,108.395062,105.005,58.083297,28.506,172.772,296.4484,462,801,2.823184
BGR,107.092007,104.350,58.789432,19.839,172.228,296.4484,454,877,2.840905


In [39]:
# --- Battery arbitrage potential: daily spread, per country ---
daily = wide.resample('D').agg(['min', 'max'])
daily.columns = [f'{c}_{stat}' for c, stat in daily.columns]

spread = pd.DataFrame({
    c: (daily[f'{c}_max'] - daily[f'{c}_min'])
    for c in countries if f'{c}_max' in daily.columns
})

fig = go.Figure()
for c in spread.columns:
    fig.add_trace(go.Box(y=spread[c], name=c))
fig.update_layout(
    title=f'Daily (max - min) price spread by country, {year}',
    yaxis_title='Spread [EUR/MWh]',
    height=500, showlegend=False,
)
fig.show()

In [40]:
# --- Single-day view for one country ---
country = 'NLD'
day = '2023-06-15'

mask = (raw['country'] == country) & (raw['t'].dt.date == pd.Timestamp(day).date())
subset = raw[mask].sort_values('t')

fig = go.Figure()
fig.add_trace(go.Scatter(x=subset['t'], y=subset['price'],
                         mode='lines+markers', name=country))
fig.update_layout(
    title=f'{country} prices on {day}',
    yaxis_title='Price [EUR/MWh]',
    height=400, hovermode='x unified',
)
fig.show()

In [10]:
# --- Compare two countries on the same day ---
countries_to_compare = ['NLD', 'DEU', 'DNK', 'FRA']
day = '2023-06-15'

mask = (raw['country'].isin(countries_to_compare)
        & (raw['t'].dt.date == pd.Timestamp(day).date()))

fig = go.Figure()
for c in countries_to_compare:
    sub = raw[mask & (raw['country'] == c)].sort_values('t')
    fig.add_trace(go.Scatter(x=sub['t'], y=sub['price'],
                             mode='lines+markers', name=c))
fig.update_layout(
    title=f'Price comparison on {day}',
    yaxis_title='Price [EUR/MWh]',
    height=400, hovermode='x unified',
)
fig.show()

In [11]:
# --- Load NL price and MDF for 2023 ---
from auxiliary_functions import load_price_from_csv, load_mdf_from_csv

price_ts = load_price_from_csv('NLD', 2023, 'ember_prices.csv')
mdf_ts = load_mdf_from_csv('mdf_nl.csv', 2023)

assert price_ts.dt == mdf_ts.dt == 1.0
n = min(len(price_ts.data), len(mdf_ts.data))
price = np.asarray(price_ts.data[:n])
mdf = np.asarray(mdf_ts.data[:n])

t = pd.date_range('2023-01-01 00:00', periods=n, freq='h')
df = pd.DataFrame({'price': price, 'mdf': mdf}, index=t)
df.describe()[['price', 'mdf']]

,price,mdf
count,8760.000000,8760.000000
mean,96.757529,208.115748
std,45.086150,93.870680
min,0.000000,-65.040476
25%,76.935000,154.685050
50%,99.215000,189.185859
75%,122.400000,245.866106
max,463.770000,1020.849726


In [12]:
# --- Weekly overlays ---
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Scatter(x=df.index, y=df['price'],
                         mode='lines', name='price [EUR/MWh]',
                         line=dict(color='royalblue', width=1)),
              secondary_y=False)
fig.add_trace(go.Scatter(x=df.index, y=df['mdf'],
                         mode='lines', name='MDF [kgCO2e/MWh]',
                         line=dict(color='firebrick', width=1)),
              secondary_y=True)

fig.update_layout(
    title='NL 2023: hourly price and MDF',
    hovermode='x unified',
    height=500,
    legend=dict(orientation='h', y=1.02, xanchor='right', x=1),
)
fig.update_yaxes(title_text='Price [EUR/MWh]', secondary_y=False)
fig.update_yaxes(title_text='MDF [kgCO2e/MWh]', secondary_y=True)
fig.show()

In [13]:
# --- Mean daily profile: hour of day ---
by_hour = df.groupby(df.index.hour).mean()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour['price'],
                         mode='lines+markers', name='price',
                         line=dict(color='royalblue')),
              secondary_y=False)
fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour['mdf'],
                         mode='lines+markers', name='MDF',
                         line=dict(color='firebrick')),
              secondary_y=True)
fig.update_layout(
    title='Mean price and MDF by hour of day, NL 2023',
    xaxis_title='Hour (UTC)',
    height=400, hovermode='x unified',
)
fig.update_yaxes(title_text='Price [EUR/MWh]', secondary_y=False)
fig.update_yaxes(title_text='MDF [kgCO2e/MWh]', secondary_y=True)
fig.show()

In [19]:
# --- Rolling correlation at different windows ---
windows = [24, 24 * 7, 24 * 30]
fig = go.Figure()
for w in windows:
    rc = df['price'].rolling(w).corr(df['mdf'])
    fig.add_trace(go.Scatter(x=df.index, y=rc, mode='lines',
                             name=f'{w}h window'))
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.update_layout(
    title='Rolling correlation between price and MDF, NL 2023',
    yaxis_title='Pearson r',
    yaxis_range=[-1, 1],
    height=450, hovermode='x unified',
)
fig.show()

In [ ]:
# Pick one week per month. Start on the first Monday of the month.
weeks = {
    'January': ('2023-01-09', '2023-01-16'),
    'April':   ('2023-04-10', '2023-04-17'),
    'July':    ('2023-07-10', '2023-07-17'),
    'October': ('2023-10-09', '2023-10-16'),
}

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    subplot_titles=[
        f"{name} ({start} to {end})"
        for name, (start, end) in weeks.items()
    ],
    specs=[[{'secondary_y': True}]] * 4,
)

for i, (name, (start, end)) in enumerate(weeks.items(), start=1):
    sub = df.loc[start:end]

    fig.add_trace(
        go.Scatter(x=sub.index, y=sub['price'],
                   mode='lines', name=f'{name} price',
                   line=dict(color='royalblue', width=1.5),
                   showlegend=(i == 1),
                   legendgroup='price'),
        row=i, col=1, secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(x=sub.index, y=sub['mdf'],
                   mode='lines', name=f'{name} MDF',
                   line=dict(color='firebrick', width=1.5),
                   showlegend=(i == 1),
                   legendgroup='mdf'),
        row=i, col=1, secondary_y=True,
    )

for i in range(1, 5):
    fig.update_yaxes(title_text='Price [EUR/MWh]',
                     secondary_y=False, row=i, col=1)
    fig.update_yaxes(title_text='MDF [kgCO2e/MWh]',
                     secondary_y=True, row=i, col=1)
    fig.update_xaxes(
        tickformat='%b %d %H:%M',
        dtick=24 * 3600 * 1000,
        tickangle=-30,
        row=i, col=1,
    )

fig.update_layout(
    height=900,
    title='NL 2023: one week per season, price and MDF',
    hovermode='x unified',
)
fig.show()

In [46]:
# --- Dutch daily mean prices, 2021-2025 ---
years = [2021, 2022, 2023, 2024, 2025]
price_csv = 'ember_prices.csv'

daily = {}
for year in years:
    try:
        ts = load_price_from_csv('NLD', year, price_csv)
    except (ValueError, FileNotFoundError) as exc:
        print(f'{year}: skipped ({exc})')
        continue
    t = pd.date_range(f'{year}-01-01 00:00', periods=len(ts.data), freq='h')
    daily[year] = pd.Series(ts.data, index=t).resample('D').mean()

print({y: len(s) for y, s in daily.items()})

{2021: 365, 2022: 365, 2023: 365, 2024: 366, 2025: 365}


In [47]:
# --- Overlay: daily mean price by year ---
fig = go.Figure()
for year, s in daily.items():
    fig.add_trace(go.Scatter(
        x=s.index.dayofyear, y=s.values,
        mode='lines', name=str(year),
    ))
fig.update_layout(
    title='NL daily mean day-ahead price, 2021-2025',
    xaxis_title='Day of year',
    yaxis_title='Daily mean price [EUR/MWh]',
    height=500, hovermode='x unified',
)
fig.show()

In [48]:
# --- Align on calendar date, one trace per year ---
fig = go.Figure()
for year, s in daily.items():
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values,
        mode='lines', name=str(year),
    ))
fig.update_layout(
    title='NL daily mean day-ahead price, 2021-2025',
    xaxis_title='Date',
    yaxis_title='Daily mean price [EUR/MWh]',
    height=500, hovermode='x unified',
)
fig.show()

In [50]:
# --- Yearly summary ---
rows = []
for year, s in daily.items():
    rows.append({
        'year': year,
        'n_days': len(s),
        'mean': s.mean(),
        'median': s.median(),
        'std': s.std(),
        'p10': s.quantile(0.10),
        'p90': s.quantile(0.90),
        'min': s.min(),
        'max': s.max(),
        'days_below_20': int((s < 20).sum()),
        'days_above_150': int((s > 150).sum()),
    })
summary = pd.DataFrame(rows).set_index('year').round(2)
summary

,n_days,mean,median,std,p10,p90,min,max,days_below_20,days_above_150
year,,,,,,,,,,
2021,365,103.09,78.19,68.16,45.94,194.11,16.34,431.99,3,84
2022,365,242.25,217.43,113.11,117.48,393.48,16.04,699.75,2,299
2023,365,96.76,97.30,32.19,53.55,137.00,4.16,202.55,4,14
2024,366,78.26,76.76,31.47,44.73,110.59,9.09,355.91,3,6
2025,365,87.62,85.54,29.10,54.79,125.61,0.50,224.40,6,10


In [52]:
# --- Box plot: distribution of daily mean price per year ---
fig = go.Figure()
for year, s in daily.items():
    fig.add_trace(go.Box(y=s.values, name=str(year), boxmean=True))
fig.update_layout(
    title='NL distribution of daily mean price, 2021-2025',
    yaxis_title='Daily mean price [EUR/MWh]',
    height=450, showlegend=False,
)
fig.show()

In [53]:
# --- Mean price by hour of day, NL, one line per year ---
years = [2021, 2022, 2023, 2024, 2025]
price_csv = 'ember_prices.csv'

by_hour_year = {}
for year in years:
    try:
        ts = load_price_from_csv('NLD', year, price_csv)
    except (ValueError, FileNotFoundError) as exc:
        print(f'{year}: skipped ({exc})')
        continue
    t = pd.date_range(f'{year}-01-01 00:00', periods=len(ts.data), freq='h')
    s = pd.Series(ts.data, index=t)
    by_hour_year[year] = s.groupby(s.index.hour).mean()

fig = go.Figure()
for year, bh in by_hour_year.items():
    fig.add_trace(go.Scatter(
        x=bh.index, y=bh.values,
        mode='lines+markers', name=str(year),
    ))
fig.update_layout(
    title='NL mean day-ahead price by hour of day, 2021-2025',
    xaxis_title='Hour (UTC)',
    yaxis_title='Mean price [EUR/MWh]',
    xaxis=dict(dtick=2),
    height=450, hovermode='x unified',
)
fig.show()

In [54]:
# --- Same data, one subplot per year, with the multi-year mean as a
# reference in each subplot ---
from plotly.subplots import make_subplots

mean_all = pd.concat(by_hour_year.values(), axis=1).mean(axis=1)

fig = make_subplots(
    rows=1, cols=len(by_hour_year),
    shared_yaxes=True,
    subplot_titles=[str(y) for y in by_hour_year],
)

for i, (year, bh) in enumerate(by_hour_year.items(), start=1):
    fig.add_trace(go.Scatter(
        x=mean_all.index, y=mean_all.values,
        mode='lines', name='2021-2025 mean',
        line=dict(color='grey', width=1, dash='dash'),
        showlegend=(i == 1),
    ), row=1, col=i)
    fig.add_trace(go.Scatter(
        x=bh.index, y=bh.values,
        mode='lines+markers', name=str(year),
        line=dict(color='royalblue'),
        showlegend=False,
    ), row=1, col=i)

fig.update_xaxes(dtick=6, title_text='Hour (UTC)')
fig.update_yaxes(title_text='Price [EUR/MWh]', col=1)
fig.update_layout(
    height=400,
    title='NL mean hourly price profile by year',
)
fig.show()

In [55]:
# --- Summary table: peak hour, trough hour, and peak-trough spread ---
rows = []
for year, bh in by_hour_year.items():
    rows.append({
        'year': year,
        'peak_hour': int(bh.idxmax()),
        'peak_price': bh.max(),
        'trough_hour': int(bh.idxmin()),
        'trough_price': bh.min(),
        'spread': bh.max() - bh.min(),
        'mean': bh.mean(),
        'p95_minus_p05': bh.quantile(0.95) - bh.quantile(0.05),
    })
pd.DataFrame(rows).set_index('year').round(2)

,peak_hour,peak_price,trough_hour,trough_price,spread,mean,p95_minus_p05
year,,,,,,,
2021,17,135.47,2,77.21,58.25,103.09,50.35
2022,17,324.48,12,183.99,140.49,242.25,127.35
2023,17,140.03,12,68.89,71.14,96.76,62.81
2024,17,123.52,12,48.52,75.00,78.26,70.32
2025,17,135.43,12,47.92,87.51,87.62,83.78
